In [1]:
# Celda 1 - Conexión e imports
from dao.mongo_dao import MongoDAO
from dao.municipio_dao import MunicipioDAO
from bson import ObjectId

mongo = MongoDAO()
db = mongo.connect()
municipio_dao = MunicipioDAO(mongo)
print("Conectado a DB:", mongo.db.name)

Conectado a DB: civictech


### 🖥️ Presentación: Listado general de municipios

Ejecuta esta celda para inicializar `MunicipioDAO` y visualizar los registros actuales de la base de datos de manera ordenada.

> 💡 **Nota visual:** El código incluye una función auxiliar que renderiza los resultados como una tabla estructurada, facilitando la lectura rápida durante la presentación.

In [2]:
# Celda 1: recargar DAO y listar municipios (presentación)
import importlib
import dao.municipio_dao as md
importlib.reload(md)
from dao.municipio_dao import MunicipioDAO

# Instanciar DAO (asume que 'mongo' está disponible en el notebook)
municipio_dao = MunicipioDAO(mongo)

def print_table(rows, headers):
    cols = list(zip(*rows)) if rows else [[] for _ in headers]
    widths = []
    for i, h in enumerate(headers):
        col = cols[i] if i < len(cols) else []
        max_cell = max([len(str(x)) for x in col], default=0)
        widths.append(max(len(h), max_cell))
    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep)
    for r in rows:
        print(" | ".join(str(r[i]).ljust(widths[i]) if i < len(r) else "".ljust(widths[i]) for i in range(len(headers))))

# Obtener municipios (sin lógica adicional)
municipios = municipio_dao.listar(limit=200)

if municipios:
    rows = []
    for m in municipios:
        rows.append([
            str(m.get("_id")),
            m.get("nombre", ""),
            m.get("provincia", ""),
            m.get("pais", ""),
            m.get("codigo_municipio", ""),
            (m.get("contacto") or {}).get("email", "")
        ])
    headers = ["_id", "nombre", "provincia", "pais", "codigo_municipio", "email_contacto"]
    print_table(rows, headers)
else:
    print("No se encontraron municipios.")


_id                      | nombre           | provincia | pais      | codigo_municipio | email_contacto            
-------------------------+------------------+-----------+-----------+------------------+---------------------------
6a28b11e3ca8ed0b349df8a3 | Famatina         | La Rioja  | Argentina | FAM001           | actualizado@contacto.local
6a28b11e3ca8ed0b349df8a4 | Chilecito        | La Rioja  | Argentina | CHL001           | transito@chilecito.gob.ar 
6a28b11e3ca8ed0b349df8a5 | La Rioja Capital | La Rioja  | Argentina | LRJ001           | transito@larioja.gob.ar   


### ➕ Insertar nuevo municipio (Validación de unicidad)

**Qué hace:** Intenta registrar un municipio de prueba ("Chamical") y maneja las excepciones devueltas por el DAO para confirmar si la operación fue exitosa o no.

* ⚠️ **Regla de negocio:** No es posible registrar el municipio si su `nombre` o su `codigo_municipio` ya existen en el sistema (el DAO bloqueará la acción devolviendo un error).

In [3]:
nuevo = {
    "nombre": "Chamical",
    "provincia": "La Rioja",
    "pais": "Argentina",
    "codigo_municipio": "PRB003",
    "contacto": {"email": "prueba@town.local", "telefono": "+54 9 0000 000000"},
    "ubicacion": {"type": "Point", "coordinates": [-67.5, -29.2]}
}

try:
    mid = municipio_dao.insertar(nuevo)
    print("✅ Municipio insertado con _id:", mid)
except ValueError as e:
    print("⚠️ No se insertó el municipio:", e)
except Exception as e:
    print("❌ Error inesperado al insertar municipio:", type(e).__name__, "-", e)

✅ Municipio insertado con _id: 6a28f4391c09f24d420d51c1


### 🔄 Previsualizar diferencias antes de actualizar

**Qué hace:** Selecciona un municipio de prueba, genera modificaciones propuestas (como cambiar el email o agregar notas) y calcula las diferencias (*diffs*) sin afectar la base de datos.

* **Salida esperada:** El estado actual del documento, el bloque de cambios sugeridos y la comparación exacta de los campos "Antes" y "Ahora".

In [4]:
# Celda: mostrar diferencias propuestas antes de actualizar
from bson import ObjectId
import json
import importlib
import dao.municipio_dao as md
importlib.reload(md)
from dao.municipio_dao import MunicipioDAO

municipio_dao = MunicipioDAO(mongo)

# tomar primer municipio
municipios = municipio_dao.listar(limit=1)
if not municipios:
    print("No hay municipios para probar.")
else:
    mid = municipios[0]["_id"]
    before = municipio_dao.obtener_por_id(mid) or {}
    cambios = {
        "contacto": {"email": "nuevo_email@ejemplo.local", "telefono": (before.get("contacto") or {}).get("telefono")},
        "nota": "Actualizado desde notebook"
    }

    def normalize(v):
        return str(v) if hasattr(v, "__class__") and v.__class__.__name__ == "ObjectId" else v

    print("Documento actual (resumen):")
    print(json.dumps({k: normalize(v) for k, v in before.items()}, ensure_ascii=False, indent=2))

    print("\nCambios propuestos:")
    print(json.dumps(cambios, ensure_ascii=False, indent=2))

    # calcular diffs simples
    diffs = []
    for k, newv in cambios.items():
        oldv = before.get(k)
        if oldv != newv:
            diffs.append((k, oldv, newv))

    if not diffs:
        print("\nResultado: No hay cambios detectados. La actualización no modificará el documento.")
    else:
        print("\nCampos que cambiarían:")
        for campo, antes, despues in diffs:
            print(f"- {campo}:")
            print("    Antes :", antes)
            print("    Ahora :", despues)



Documento actual (resumen):
{
  "_id": "6a28b11e3ca8ed0b349df8a3",
  "nombre": "Famatina",
  "provincia": "La Rioja",
  "pais": "Argentina",
  "codigo_municipio": "FAM001",
  "contacto": {
    "email": "actualizado@contacto.local",
    "telefono": "+54 3825 111111"
  },
  "nota": "Actualizado desde notebook"
}

Cambios propuestos:
{
  "contacto": {
    "email": "nuevo_email@ejemplo.local",
    "telefono": "+54 3825 111111"
  },
  "nota": "Actualizado desde notebook"
}

Campos que cambiarían:
- contacto:
    Antes : {'email': 'actualizado@contacto.local', 'telefono': '+54 3825 111111'}
    Ahora : {'email': 'nuevo_email@ejemplo.local', 'telefono': '+54 3825 111111'}


### 🗑️ Eliminar municipio por código

**Qué hace:** Elimina un municipio utilizando su `codigo_municipio` a través del método correspondiente del DAO. 

* **Salida esperada:** Muestra el `_id` y el `nombre` del municipio afectado para confirmar la eliminación.

In [5]:
# Celda: eliminar municipio por codigo_municipio mostrando _id y nombre (usa el método del DAO)
import importlib
import dao.municipio_dao as md
importlib.reload(md)
from dao.municipio_dao import MunicipioDAO
from bson import ObjectId
import json

municipio_dao = MunicipioDAO(mongo)

def banner(msg, kind="INFO"):
    line = "=" * (len(msg) + 8)
    tag = "!!!" if kind == "ERROR" else "***"
    print(f"\n{line}\n{tag}  {msg}  {tag}\n{line}\n")

def safe_str(v):
    try:
        return str(v)
    except Exception:
        return repr(v)

# Ajustá este código si querés otro municipio
codigo = "PRB003"

banner(f"ELIMINAR MUNICIPIO por codigo_municipio = {codigo}", kind="INFO")

# Obtener documento antes de intentar eliminar para mostrar nombre
doc = municipio_dao.obtener_por_codigo(codigo)
if not doc:
    banner(f"No se encontró ningún municipio con codigo_municipio = {codigo}", kind="ERROR")
else:
    mid_before = doc.get("_id")
    nombre_before = doc.get("nombre")
    print("Municipio encontrado (pre-borrado):")
    print("  _id   :", safe_str(mid_before))
    print("  nombre:", nombre_before)
    print("  provincia:", doc.get("provincia"))
    print("  pais     :", doc.get("pais"))
    print("\nIntentando eliminación usando el DAO...")

    try:
        result = municipio_dao.eliminar_por_codigo(codigo)
        if result.get("success"):
            banner("✅ Municipio eliminado correctamente", kind="INFO")
            print("  municipio_id:", result.get("municipio_id"))
            print("  nombre      :", nombre_before)
            print("  mensaje     :", result.get("message"))
            print("\nVerificación sugerida (mongo shell):")
            print(f"  mongo.db.municipio.findOne({{'_id': ObjectId('{safe_str(result.get('municipio_id'))}')}})")
        else:
            action = result.get("action")
            if action == "not_found":
                banner("⚠️ Municipio no encontrado al intentar eliminar", kind="ERROR")
                print(result.get("message"))
            else:
                banner("⚠️ Operación no completada", kind="ERROR")
                print("action  :", action)
                print("message :", result.get("message"))
    except ValueError as ve:
        banner("⛔ Operación bloqueada por regla de negocio", kind="ERROR")
        print("Motivo:", ve)
        print("\nAcción recomendada:")
        print("  - Inspeccionar usuarios asociados:")
        print(f"    mongo.db.usuario.find({{'municipio_id': ObjectId('{safe_str(mid_before)}')}}).limit(10).pretty()")
        print("  - Eliminar o reasignar usuarios antes de reintentar el borrado.")
    except Exception as e:
        banner("❌ Error inesperado al intentar eliminar", kind="ERROR")
        print(type(e).__name__, "-", e)



***  ELIMINAR MUNICIPIO por codigo_municipio = PRB003  ***

Municipio encontrado (pre-borrado):
  _id   : 6a28f4391c09f24d420d51c1
  nombre: Chamical
  provincia: La Rioja
  pais     : Argentina

Intentando eliminación usando el DAO...

***  ✅ Municipio eliminado correctamente  ***

  municipio_id: 6a28f4391c09f24d420d51c1
  nombre      : Chamical
  mensaje     : Municipio eliminado correctamente

Verificación sugerida (mongo shell):
  mongo.db.municipio.findOne({'_id': ObjectId('6a28f4391c09f24d420d51c1')})


In [6]:
# Celda 5 - Borrar municipio con chequeo de usuarios asociados
# Selecciona el primer municipio disponible si no se provee un _id explícito
municipios = municipio_dao.listar(limit=10)
mid = municipios[0]["_id"] if municipios else None

def print_doc_vertical(doc):
    if not doc:
        print("Documento vacío o no encontrado")
        return
    for k, v in doc.items():
        print(f"{k}: {v}")

if not mid:
    print("No hay municipios disponibles para borrar.")
else:
    print("Municipio seleccionado (_id):", mid)
    doc = municipio_dao.obtener_por_id(mid)
    print("\nDocumento:")
    print_doc_vertical(doc)

    try:
        usuarios_count = municipio_dao.contar_usuarios(mid)
        print("\nUsuarios asociados:", usuarios_count)
        if usuarios_count == 0:
            deleted = municipio_dao.borrar(mid)
            if deleted:
                print("\n✅ Municipio borrado correctamente (deleted_count = 1).")
            else:
                print("\n⚠️ Intento de borrado realizado pero deleted_count = 0 (posible condición de carrera).")
        else:
            print("\n⛔ No se borra: existen usuarios asociados.")
    except ValueError as e:
        print("\n⚠️ Error de validación:", e)
    except Exception as e:
        print("\n❌ Error inesperado al intentar borrar:", type(e).__name__, "-", e)


Municipio seleccionado (_id): 6a28b11e3ca8ed0b349df8a3

Documento:
_id: 6a28b11e3ca8ed0b349df8a3
nombre: Famatina
provincia: La Rioja
pais: Argentina
codigo_municipio: FAM001
contacto: {'email': 'actualizado@contacto.local', 'telefono': '+54 3825 111111'}
nota: Actualizado desde notebook

Usuarios asociados: 5

⛔ No se borra: existen usuarios asociados.


In [7]:
# Celda eliminar municipio por codigo_municipio
codigo = "PRB002"   # ajustá si querés otro código

# Buscar documento
doc = municipio_dao.obtener_por_codigo(codigo)
if not doc:
    print(f"No se encontró ningún municipio con codigo_municipio = {codigo}")
else:
    mid = doc["_id"]
    print("Municipio encontrado:")
    print("  _id:", mid)
    print("  nombre:", doc.get("nombre"))
    print("  provincia:", doc.get("provincia"))
    print("  codigo_municipio:", doc.get("codigo_municipio"))

    try:
        usuarios_count = municipio_dao.contar_usuarios(mid)
        print("Usuarios asociados:", usuarios_count)
        if usuarios_count == 0:
            deleted = municipio_dao.borrar(mid)
            if deleted:
                print("✅ Municipio eliminado correctamente (deleted_count = 1).")
            else:
                print("⚠️ Intento de borrado realizado pero deleted_count = 0.")
        else:
            print("⛔ No se borra: existen usuarios asociados.")
    except ValueError as e:
        print("⚠️ Error de validación:", e)
    except Exception as e:
        print("❌ Error inesperado al intentar borrar:", type(e).__name__, "-", e)


No se encontró ningún municipio con codigo_municipio = PRB002
